# Tensor Operations Notebook

> Hands-on Build It and Exercises.

## Build It

The code lives in `code/tensors.py`. Each step references the implementation there.

### Step 1: Tensor storage and strides

A tensor stores a flat list of numbers plus shape metadata. Strides tell the indexing logic how to map multi-dimensional indices to flat positions.

In [ ]:
```python

class Tensor:

    def __init__(self, data, shape=None):

        if isinstance(data, (list, tuple)):

            self._data, self._shape = self._flatten_nested(data)

        elif isinstance(data, np.ndarray):

            self._data = data.flatten().tolist()

            self._shape = tuple(data.shape)

        else:

            self._data = [data]

            self._shape = ()

        if shape is not None:

            total = reduce(lambda a, b: a * b, shape, 1)

            if total != len(self._data):

                raise ValueError(

                    f"Cannot reshape {len(self._data)} elements into shape {shape}"

                )

            self._shape = tuple(shape)

        self._strides = self._compute_strides(self._shape)

    @staticmethod

    def _compute_strides(shape):

        if len(shape) == 0:

            return ()

        strides = [1] * len(shape)

        for i in range(len(shape) - 2, -1, -1):

            strides[i] = strides[i + 1] * shape[i + 1]

        return tuple(strides)

In [ ]:
```

For shape `(3, 4)`, strides are `(4, 1)` -- skip 4 elements to advance one row, skip 1 element to advance one column.

### Step 2: Reshape, squeeze, unsqueeze

Reshape changes the shape without changing element order. The total number of elements must stay the same. Use `-1` for one dimension to infer its size.

In [ ]:
```python

t = Tensor(list(range(12)), shape=(2, 6))

r = t.reshape((3, 4))

r = t.reshape((-1, 3))

In [ ]:
```

Squeeze removes axes of size 1. Unsqueeze inserts one. Unsqueezing is critical for broadcasting -- a bias vector `(D,)` added to a batch `(B, T, D)` needs unsqueezing to `(1, 1, D)`.

In [ ]:
```python

t = Tensor(list(range(6)), shape=(1, 3, 1, 2))

s = t.squeeze()

v = Tensor([1, 2, 3])

u = v.unsqueeze(0)

In [ ]:
```

### Step 3: Transpose and permute

Transpose swaps two axes. Permute reorders all axes. This is how you convert between NCHW and NHWC.

In [ ]:
```python

mat = Tensor(list(range(6)), shape=(2, 3))

tr = mat.transpose(0, 1)

t4d = Tensor(list(range(24)), shape=(1, 2, 3, 4))

perm = t4d.permute((0, 2, 3, 1))

In [ ]:
```

After transpose or permute, the tensor is non-contiguous in memory. In PyTorch, `view` fails on non-contiguous tensors -- use `reshape` or call `.contiguous()` first.

### Step 4: Element-wise operations and reductions

Element-wise ops (add, multiply, subtract) apply independently to each element and preserve shape. Reductions (sum, mean, max) collapse one or more axes.

In [ ]:
```python

a = Tensor([[1, 2], [3, 4]])

b = Tensor([[10, 20], [30, 40]])

c = a + b

d = a * 2

s = a.sum(axis=0)

In [ ]:
```

Global average pooling in a CNN: `(B, C, H, W).mean(axis=[2, 3])` produces `(B, C)`. Sequence mean pooling in NLP: `(B, T, D).mean(axis=1)` produces `(B, D)`.

### Step 5: Broadcasting with NumPy

The `demo_broadcasting_numpy()` function in `tensors.py` shows the core patterns.

In [ ]:
```python

activations = np.random.randn(4, 3)

bias = np.array([0.1, 0.2, 0.3])

result = activations + bias

images = np.random.randn(2, 3, 4, 4)

scale = np.array([0.5, 1.0, 1.5]).reshape(1, 3, 1, 1)

result = images * scale

a = np.array([1, 2, 3]).reshape(-1, 1)

b = np.array([10, 20, 30, 40]).reshape(1, -1)

outer = a * b

In [ ]:
```

Pairwise distance via broadcasting: reshape `(M, 2)` to `(M, 1, 2)` and `(N, 2)` to `(1, N, 2)`, subtract, square, sum along last axis, take square root. Result: `(M, N)`.

### Step 6: Einsum operations

The `demo_einsum()` and `demo_einsum_gallery()` functions walk through every common pattern.

In [ ]:
```python

a = np.array([1.0, 2.0, 3.0])

b = np.array([4.0, 5.0, 6.0])

dot = np.einsum("i,i->", a, b)

A = np.array([[1, 2], [3, 4], [5, 6]], dtype=float)

B = np.array([[7, 8, 9], [10, 11, 12]], dtype=float)

matmul = np.einsum("ik,kj->ij", A, B)

batch_A = np.random.randn(4, 3, 5)

batch_B = np.random.randn(4, 5, 2)

batch_mm = np.einsum("bij,bjk->bik", batch_A, batch_B)

In [ ]:
```

The computational cost of a contraction is the product of all index sizes (kept and summed). For `bij,bjk->bik` with B=32, I=128, J=64, K=128: `32 * 128 * 64 * 128 = 33,554,432` multiply-adds.

### Step 7: Attention mechanism via einsum

The `demo_attention_einsum()` function implements multi-head attention end to end.

In [ ]:
```python

B, H, T, D = 2, 4, 8, 16

E = H * D

X = np.random.randn(B, T, E)

W_q = np.random.randn(E, E) * 0.02

Q = np.einsum("bte,ek->btk", X, W_q)

Q = Q.reshape(B, T, H, D).transpose(0, 2, 1, 3)

scores = np.einsum("bhtd,bhsd->bhts", Q, K) / np.sqrt(D)

weights = softmax(scores, axis=-1)

attn_output = np.einsum("bhts,bhsd->bhtd", weights, V)

concat = attn_output.transpose(0, 2, 1, 3).reshape(B, T, E)

output = np.einsum("bte,ek->btk", concat, W_o)

In [ ]:
```

Every step is a tensor operation: projection (matmul via einsum), head splitting (reshape + transpose), attention scores (batch matmul via einsum), weighted sum (batch matmul via einsum), head merging (transpose + reshape), output projection (matmul via einsum).

## Exercises

In [ ]:
1. **Easy -- Reshape round-trip.** Take a tensor of shape `(2, 3, 4)`. Reshape it to `(6, 4)`, then to `(24,)`, then back to `(2, 3, 4)`. Verify element order is preserved at each step by printing the flat data.

2. **Medium -- Implement broadcasting.** Extend the `Tensor` class with a `broadcast_to(shape)` method that expands dimensions of size 1 to match a target shape. Then modify `_elementwise_op` to automatically broadcast before operating. Test with shapes `(3, 1)` and `(1, 4)` producing `(3, 4)`.

3. **Hard -- Build einsum from scratch.** Implement a basic `einsum(subscripts, *tensors)` function that handles at least: dot product (`i,i->`), matrix multiply (`ij,jk->ik`), outer product (`i,j->ij`), and transpose (`ij->ji`). Parse the subscript string, identify contracted indices, and loop over all index combinations. Compare your results against `np.einsum`.

4. **Hard -- Attention shape tracker.** Write a function that takes `batch_size`, `seq_len`, `embed_dim`, and `num_heads` as inputs and prints the exact shape at every step of multi-head attention: input, Q/K/V projection, head split, attention scores, softmax weights, weighted sum, head merge, output projection. Verify against the `demo_attention_einsum()` output.